# Stat 451 – Group 12: Spotify Tracks Data Cleaning

**Authors:** Arianna Barbosa, Eirfann Danish Bin Farul Muzri, Mukhriz Hazli Bin Mohamad, Greta Fogel, Levi Hellenbrand, Lexi Rush

**Dataset source:** [Kaggle – Spotify Tracks Dataset](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)

---

## Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 120)

## 2. Load the Raw Dataset

In [2]:
#raw csv file
df_raw = pd.read_csv('dataset.csv')

print(f'Raw dataset shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head()

Raw dataset shape: 114,000 rows × 21 columns


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## FInd out how many nulls

In [3]:
# Data types and null counts
print('Null Counts')
print(df_raw.isnull().sum())
print()

Null Counts
Unnamed: 0          0
track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64



## 4. Data Cleaning


In [4]:
df = df_raw.copy()
print(f'Starting shape: {df.shape}')

Starting shape: (114000, 21)


### Drop the Unnamed: 0 Column

The Unnamed column is just the numbers of tracks, not important at all. Not informative

In [5]:
# only run it one time
df.drop(columns=['Unnamed: 0'], inplace=True)
print(f'{df.shape[1]} columns remain')

20 columns remain


### Handle Missing Values

There is exactly 1 row where artists, album_name, and track_name are all null.

In [6]:
# No imputation cus we cant figure out what is the missing songs/track
df.dropna(inplace=True)
print(f'After dropping nulls: {df.shape[0]:,} rows')

After dropping nulls: 113,999 rows


### Handling Duplicated track_id s

Many tracks appear in multiple rows because the same song is tagged under different genres. example a rock song might appear in alt-rock and once again under rock.

If we leave duplicates in, the model would see the same audio features multiple times, which biases the model toward tracks that happen to belong to many genres.

In [7]:
before = len(df)
df.drop_duplicates(subset='track_id', keep='first', inplace=True)
after = len(df)
print(f'Removed {before - after:,} duplicate track_id rows')
print(f'Remaining rows: {after:,}')

Removed 24,259 duplicate track_id rows
Remaining rows: 89,740


### Filter Out Non-Music Tracks (Speechiness > 0.66)

According to the documentation:
- speechiness > 0.66 = tracks that are probably made entirely of spoken words (talk shows, audiobooks, poetry)
- speechiness 0.33–0.66 = tracks with both music and speech (e.g., rap)
- speechiness < 0.33 = music and other non-speech tracks

Since our project focuses on predicting the popularity of music, tracks above 0.66 are likely not songs at all. We remove them. We keep the <= 0.66 range because it includes legitimate music genres like rap and hip-hop.

In [8]:
before = len(df)
high_speech = (df['speechiness'] > 0.66).sum()
print(f'Tracks with speechiness > 0.66: {high_speech}')

df = df[df['speechiness'] <= 0.66]
print(f'Removed {before - len(df):,} word tracks')
print(f'Remaining rows: {len(df):,}')

Tracks with speechiness > 0.66: 868
Removed 868 word tracks
Remaining rows: 88,872


### Remove Zero or Near-Zero Duration Tracks

A track with 0 ms duration is clearly an error

We also remove tracks shorter than 30 seconds (30,000 ms), as these are typically intro clips, sound effects, or metadata artifacts rather than complete songs.

In [9]:
before = len(df)
print(f'Tracks with duration = 0 ms: {(df["duration_ms"] == 0).sum()}')
print(f'Tracks with duration < 30s: {(df["duration_ms"] < 30000).sum()}')

df = df[df['duration_ms'] >= 30000]
print(f'Removed {before - len(df):,} extremely short tracks')
print(f'Remaining rows: {len(df):,}')

Tracks with duration = 0 ms: 0
Tracks with duration < 30s: 13
Removed 13 extremely short tracks
Remaining rows: 88,859


### Remove Tracks with Zero Tempo

Since tempo is one of our predictive features, keeping rows would just be noises. songs with 0 tempo is just a few so we should still be fine

In [10]:
before = len(df)
print(f'Tracks with tempo = 0: {(df["tempo"] == 0).sum()}')

df = df[df['tempo'] > 0]
print(f'Removed {before - len(df):,} zero-tempo tracks')
print(f'Remaining rows: {len(df):,}')

Tracks with tempo = 0: 155
Removed 155 zero-tempo tracks
Remaining rows: 88,704


### Converting explicit from Boolean to Integer

Our ML will most likely use 0 or 1, so I am encoding the value from TRUE/FALSE to 1/0.

In [11]:
df['explicit'] = df['explicit'].astype(int)
print('explicit dtype:', df['explicit'].dtype)
print(df['explicit'].value_counts())

explicit dtype: int32
explicit
0    81534
1     7170
Name: count, dtype: int64


### Convert duration_ms to duration_min

it is just easier to understands

Converting to minutes makes the feature more interpretable. We drop the original duration_ms column afterward. 

In [12]:
df['duration_min'] = df['duration_ms'] / 60000.0
df.drop(columns=['duration_ms'], inplace=True)
print(f'duration_min range: {df["duration_min"].min():.2f} – {df["duration_min"].max():.2f} minutes')

duration_min range: 0.50 – 87.29 minutes


### 4.9 Drop Non-Feature Metadata Columns

The columns track_id, artists, album_name, and track_name are identifiers/metadata. They are not audio features and should NOT be used as predictors in our models:

- track_id is a unique id, no predictive meaning.
- artists and album_name. would cause overfitting and doesn't align with our question (not whether being a famous artist predicts popularity, bias towards famous artist).
- track_name is unique and uninformative for modeling.



In [13]:
metadata_cols = ['track_id', 'artists', 'album_name', 'track_name']
df.drop(columns=metadata_cols, inplace=True)
print(f'Dropped {metadata_cols}')
print(f'Remaining columns: {list(df.columns)}')

Dropped ['track_id', 'artists', 'album_name', 'track_name']
Remaining columns: ['popularity', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre', 'duration_min']


### Reset the Index

After filtering, the DataFrame index has changes. reset the gaps in order. This avoids indexing confusion later.

In [14]:
df.reset_index(drop=True, inplace=True)
print(f'Index reset. Final shape: {df.shape}')

Index reset. Final shape: (88704, 16)


## Export Cleaned Dataset

USE THIS FOR ALL THE PREDICTIVE MODELLINGS AND ML

In [15]:
output_path = 'spotify_cleaned.csv'
df.to_csv(output_path, index=False)
print(f'Cleaned dataset: {output_path}')
print(f'Final clean shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

Cleaned dataset: spotify_cleaned.csv
Final clean shape: 88,704 rows × 16 columns
